# Crop yield model

Simulate the yield trajectory of each crop under three regimes: watered daily, watered + fertilized during the bonus window, and skipped (not watered). Report expected coins per tile per day at the base sell price.

One-time crops (wheat, carrot, melon): watering during the bonus window (starting at `ceil(max_yield_day / 2)`) adds one unit to harvestable yield per day. Fertilizer doubles that bonus for three days. Ongoing crops (tomato, strawberry) fire scheduled productions at fixed intervals: base 1, or 2 if fertilized AND watered that day.

After max lifespan the plant decays by 1 unit every other turn until it becomes a weed.

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from kaggriculture.env.constants import CROPS, MARKET_PARAMS

FIG_DIR = Path.cwd().parent / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def one_time_yield(
    crop: str, days: int, watered: bool = True, fertilized: bool = False
) -> list[int]:
    """Trace yield_units across `days` for a one-time crop under a given care regime."""
    cd = CROPS[crop]
    assert not cd["ongoing"], crop
    window_start = (cd["max_yield_day"] + 1) // 2
    yields = []
    y = 1
    for day in range(days + 1):
        if watered and window_start <= day <= cd["max_yield_day"]:
            bonus = 2 if fertilized else 1
            y = min(cd["max_yield"], y + bonus)
        if day > cd["max_yield_day"] + 1 and (day - cd["max_yield_day"] - 1) % 2 == 0:
            y = max(0, y - 1)
        yields.append(y)
    return yields

In [ ]:
def ongoing_cumulative(
    crop: str, days: int, watered: bool = True, fertilized: bool = False
) -> list[int]:
    """Cumulative units produced over `days` for an ongoing crop."""
    cd = CROPS[crop]
    assert cd["ongoing"], crop
    cum = 0
    total_prod = 0
    trace = []
    for day in range(days + 1):
        days_since_first = day - cd["first_yield_day"]
        produce = (
            watered
            and days_since_first >= 0
            and days_since_first % cd["interval"] == 0
            and total_prod < cd["max_yield"]
        )
        if produce:
            bonus = 1 if fertilized else 0
            cum += 1 + bonus
            total_prod += 1
        trace.append(cum)
    return trace

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, crop in zip(axes.flat, CROPS, strict=False):
    cd = CROPS[crop]
    horizon = cd["max_yield_day"] + 6
    days = np.arange(horizon + 1)
    if cd["ongoing"]:
        ax.plot(
            days,
            ongoing_cumulative(crop, horizon, watered=True, fertilized=False),
            label="watered",
            color="#1f77b4",
        )
        ax.plot(
            days,
            ongoing_cumulative(crop, horizon, watered=True, fertilized=True),
            label="watered + fertilized",
            color="#2ca02c",
        )
        ax.plot(
            days,
            ongoing_cumulative(crop, horizon, watered=False, fertilized=False),
            label="skipped",
            color="#d62728",
            linestyle=":",
        )
        ax.set_ylabel("cumulative units")
    else:
        ax.plot(
            days,
            one_time_yield(crop, horizon, watered=True, fertilized=False),
            label="watered",
            color="#1f77b4",
        )
        ax.plot(
            days,
            one_time_yield(crop, horizon, watered=True, fertilized=True),
            label="watered + fertilized",
            color="#2ca02c",
        )
        ax.plot(
            days,
            one_time_yield(crop, horizon, watered=False, fertilized=False),
            label="skipped",
            color="#d62728",
            linestyle=":",
        )
        ax.set_ylabel("yield_units on plant")
    ax.axvline(cd["first_yield_day"], color="gray", linestyle="--", linewidth=0.7)
    ax.axvline(cd["max_yield_day"], color="black", linestyle=":", linewidth=0.7)
    ax.set_title(f"{crop}  (seed=${cd['seed']}, base=${MARKET_PARAMS[crop]['base']})")
    ax.set_xlabel("day")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

axes.flat[-1].axis("off")
fig.suptitle("Crop yield trajectories under three care regimes", y=1.00)
fig.tight_layout()
fig.savefig(FIG_DIR / "crop-yields.png", dpi=140, bbox_inches="tight")
plt.show()

In [ ]:
def peak_units_and_days(crop: str, fertilized: bool) -> tuple[int, int]:
    cd = CROPS[crop]
    if not cd["ongoing"]:
        y = one_time_yield(crop, cd["max_yield_day"], watered=True, fertilized=fertilized)
        peak = y[-1]
        return peak, cd["max_yield_day"]
    max_intervals = cd["max_yield"] - 1
    last_prod_day = cd["first_yield_day"] + max_intervals * cd["interval"]
    cum = ongoing_cumulative(crop, last_prod_day, watered=True, fertilized=fertilized)
    return cum[-1], last_prod_day


rows = []
for crop in CROPS:
    for fertilized in (False, True):
        units, days = peak_units_and_days(crop, fertilized)
        base_price = MARKET_PARAMS[crop]["base"]
        revenue = units * base_price
        cost = CROPS[crop]["seed"]
        net_per_tile_day = (revenue - cost) / max(1, days)
        rows.append(
            {
                "crop": crop,
                "fertilized": fertilized,
                "peak_units": units,
                "days_to_peak": days,
                "gross_at_base": revenue,
                "net_at_base": revenue - cost,
                "$/tile/day": round(net_per_tile_day, 2),
            }
        )

df = pd.DataFrame(rows).set_index(["crop", "fertilized"])
df

In [ ]:
flat = df.reset_index()
fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(CROPS))
width = 0.35
no_fert = flat[~flat["fertilized"]].set_index("crop").reindex(CROPS)
yes_fert = flat[flat["fertilized"]].set_index("crop").reindex(CROPS)
ax.bar(x - width / 2, no_fert["$/tile/day"], width, label="watered only", color="#1f77b4")
ax.bar(x + width / 2, yes_fert["$/tile/day"], width, label="watered + fertilized", color="#2ca02c")
ax.set_xticks(x)
ax.set_xticklabels(list(CROPS))
ax.set_ylabel("$ per tile per day (at base price)")
ax.set_title("Net revenue per tile per day at base sell price")
ax.axhline(0, color="black", linewidth=0.8)
ax.grid(True, axis="y", alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "crop-roi.png", dpi=140, bbox_inches="tight")
plt.show()

## Takeaways

At base prices with optimal watering:

- Melon is the highest $/tile/day one-time crop by far, thanks to the $250 base and 6-unit max yield. It takes 10 days to reach peak, so it locks tiles and is only viable if the market absorbs it (see notebook 01: melon crashes to $1 on modest glut).
- Strawberry and tomato both cap at 4 scheduled productions; with fertilizer+water they double each. Strawberry's every-other-day interval gives it a longer occupancy but higher per-unit price.
- Carrot is the fastest ROI: 3-day cycle, $35 base, seed only $20. Ideal for high-turnover play.
- Wheat is a low-margin staple. Its value comes from feeding animals, not selling.
- Fertilizer materially improves every crop's $/tile/day at base prices. Its own market cost ($100 to buy) has to be weighed, but fertilizer produced by animals is essentially free.